In [ ]:
!pip install -q transformers langchain langchain-core langchain-huggingface huggingface-hub accelerate

In [ ]:
from langchain_huggingface import HuggingFacePipeline

from transformers import AutoTokenizer
import transformers

# This line imports the torch library, which is the primary library used for deep learning and tensor computations in PyTorch.
import torch

#model = "meta-llama/Llama-2-7b-chat-hf"
model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model)

# Set up text generation pipeline
pipeline = transformers.pipeline("text-generation",
                model=model,
                tokenizer= tokenizer,
                dtype=torch.bfloat16, # Changed torch_dtype to dtype
                device_map="auto",
                max_new_tokens = 512,
                do_sample=True,
                top_k=10,
                num_return_sequences=1,
                eos_token_id=tokenizer.eos_token_id,
                )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# 'HuggingFacePipeline' class creates a custom pipeline for text generation, and we are passing
# the pipeline that we defined earlier along with some model-specific keyword arguments - temperature here.

llm = HuggingFacePipeline(pipeline = pipeline, model_kwargs = {'temperature':0})

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
             Create a SQL query snippet using the below text:
              ```{text}```
              10  SQL query:
           """

prompt = PromptTemplate(template=template, input_variables=["text"])

# llm_chain = LLMChain(prompt=prompt, llm=llm)
# Going forward we might have to use the below code snippet instead of above:
llm_chain = prompt | llm

text = """ Extract all the unique values from column "age"
like limts , groupby etc.
"""


In [ ]:
print(llm_chain.invoke(text))

Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



             Create a SQL query snippet using the below text:
              ``` Extract all the unique values from column "age"
like limts , groupby etc. 
```
              10  SQL query:
            ``` SQL
            WITH age AS(
                SELECT DISTINCT CAST(age AS INT) AS age
                FROM my_table
                GROUP BY age
            )
            SELECT age, COUNT(*)
            FROM age
            GROUP BY age;
            ```
            ``` SQL
            SELECT age, COUNT(*) FROM age GROUP BY age;
            ```
            ``` SQL
            SELECT age
            FROM (
                SELECT DISTINCT CAST(age AS INT) AS age
                FROM my_table
                GROUP BY age
            ) AS age_group;
            ```
            ``` SQL
            SELECT age, COUNT(*)
            FROM (
                SELECT DISTINCT CAST(age AS INT) AS age
                FROM my_table
                GROUP BY age
            ) AS age_group
            GR